# Entrenamiento de VotoInteligente | ResNet50
Vamos a entrenar con los dibujos del canvas y comparar dos fases.

## 1. Librerías
Activa una GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución.

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense, Dropout,
                                     BatchNormalization, RandomRotation,
                                     RandomTranslation, RandomZoom)
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Subir los dibujos
Comprime dataset/ con sus cuatro carpetas y PNG como dataset.zip y súbelo aquí.

In [ ]:
from google.colab import files

subidos = files.upload()
archivo_zip = next((name for name in subidos if name.lower().endswith('.zip')), None)
if archivo_zip is None:
    raise ValueError('Sube dataset.zip.')
carpeta = Path('/content/votointeligente_datos')
if carpeta.exists():
    shutil.rmtree(carpeta)
carpeta.mkdir()
with zipfile.ZipFile(archivo_zip) as zf:
    for nombre in zf.namelist():
        destino = (carpeta / nombre).resolve()
        if not destino.is_relative_to(carpeta.resolve()):
            raise ValueError('El ZIP contiene una ruta no válida.')
    zf.extractall(carpeta)
dataset_dir = carpeta / 'dataset' if (carpeta / 'dataset').is_dir() else carpeta
esperadas = ['aspa_valida', 'cruz_valida', 'interseccion_fuera', 'marca_diferente']
if not all((dataset_dir / clase).is_dir() for clase in esperadas):
    raise ValueError('Faltan las cuatro carpetas de categorías en el ZIP.')
print('Carpeta:', dataset_dir)

## 3. Leer etiquetas
Aquí contamos los PNG de cada categoría, en orden alfabético.

In [ ]:
clases = sorted(esperadas)
archivos = []
etiquetas = []
for indice, clase in enumerate(clases):
    imagenes = sorted((dataset_dir / clase).glob('*.png'))
    print(clase, ':', len(imagenes), 'dibujos')
    for path in imagenes:
        archivos.append(str(path))
        etiquetas.append(indice)
if any(etiquetas.count(i) < 10 for i in range(len(clases))):
    raise ValueError('Se necesitan al menos 10 PNG por categoría.')
print('Total:', len(archivos))
plt.figure(figsize=(8, 3))
plt.bar(clases, [etiquetas.count(i) for i in range(4)])
plt.xticks(rotation=15)
plt.title('Dibujos por categoría')
plt.tight_layout()
plt.show()

## 4. Ver ejemplos
Mostramos un dibujo de cada categoría.

In [ ]:
from tensorflow.keras.preprocessing import image

plt.figure(figsize=(10, 3))
for i, clase in enumerate(clases):
    archivo = sorted((dataset_dir / clase).glob('*.png'))[0]
    img = image.load_img(archivo, target_size=(200, 200))
    plt.subplot(1, 4, i + 1)
    plt.imshow(img)
    plt.title(clase)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Separar los datos
Usamos 70/15/15 con semilla 42; la prueba queda fuera del entrenamiento. Como todos los dibujos son "anonimo", no se puede separar por persona y la prueba puede salir algo optimista.

In [ ]:
archivos = np.array(archivos)
etiquetas = np.array(etiquetas)
x_train, x_resto, y_train, y_resto = train_test_split(
    archivos, etiquetas, test_size=0.30, random_state=42, stratify=etiquetas)
x_val, x_test, y_val, y_test = train_test_split(
    x_resto, y_resto, test_size=0.50, random_state=42, stratify=y_resto)
print('Train:', len(x_train), 'Validación:', len(x_val), 'Prueba:', len(x_test))
pesos = compute_class_weight('balanced', classes=np.arange(len(clases)), y=y_train)
class_weight = dict(enumerate(pesos))
print('Pesos:', class_weight)

## 6. Preparar imágenes
El aumento se aplica solo en train, antes de preprocess_input y fuera del modelo. Giro ±5°, traslación ±5% y zoom ±8%, con relleno blanco y sin cortar el recuadro.

In [ ]:
target_size = (224, 224)
batch_size = 16
augment = Sequential([
    RandomRotation(5 / 360, fill_mode='constant', fill_value=255, seed=42),
    RandomTranslation(0.05, 0.05, fill_mode='constant', fill_value=255, seed=43),
    RandomZoom((-0.08, 0.08), width_factor=(-0.08, 0.08),
               fill_mode='constant', fill_value=255, seed=44),
])

def leer_imagen(ruta, etiqueta):
    contenido = tf.io.read_file(ruta)
    imagen = tf.io.decode_png(contenido, channels=3)
    imagen = tf.image.resize(imagen, target_size)
    return tf.cast(imagen, tf.float32), etiqueta

def crear_dataset(x, y, shuffle=False, aumentar=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(len(x), seed=42)
    ds = ds.map(leer_imagen, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    def preparar(imagenes, etiquetas):
        if aumentar:
            imagenes = augment(imagenes, training=True)
        return preprocess_input(imagenes), etiquetas
    ds = ds.map(preparar, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = crear_dataset(x_train, y_train, shuffle=True, aumentar=True)
train_eval_ds = crear_dataset(x_train, y_train)
val_ds = crear_dataset(x_val, y_val)
test_ds = crear_dataset(x_test, y_test)

## 7. Crear la red
Usamos ResNet50 con pesos de ImageNet, dropout de 0.4 y L2 de 1e-4.

In [ ]:
base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False
model = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dropout(0.4),
    Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
    Dense(len(clases), activation='softmax'),
])
model.build((None, 224, 224, 3))
model.summary()

## 8. Preparar callbacks
Medimos la brecha con train sin aumento ni dropout para compararlo con validación. Guardamos el mínimo de val_loss y otro mínimo que cumpla una brecha absoluta ≤5 puntos.

In [ ]:
carpeta_resultados = Path('/content/resultados_entrenamiento')
carpeta_resultados.mkdir(exist_ok=True)
brecha_maxima = 0.05
accuracy_minima = 0.80  # referencia ajustable para detectar subajuste

class MedirTrain(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        medidas = self.model.evaluate(train_eval_ds, verbose=0, return_dict=True)
        logs['train_eval_accuracy'] = medidas['accuracy']
        logs['train_eval_loss'] = medidas['loss']

class CheckpointConBrecha(tf.keras.callbacks.ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        brecha = abs(logs['train_eval_accuracy'] - logs['val_accuracy'])
        if brecha <= brecha_maxima:
            super().on_epoch_end(epoch, logs)

def callbacks(fase):
    return [
        MedirTrain(),
        tf.keras.callbacks.ModelCheckpoint(
            str(carpeta_resultados / f'{fase}_mejor.keras'),
            monitor='val_loss', save_best_only=True),
        CheckpointConBrecha(
            str(carpeta_resultados / f'{fase}_brecha.keras'),
            monitor='val_loss', save_best_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=4, restore_best_weights=True),
    ]

# evita reutilizar checkpoints de otra ejecución
for ruta in carpeta_resultados.glob('fase*.keras'):
    ruta.unlink()

## 9. Fase 1
Entrenamos el clasificador con la base congelada, hasta 20 épocas.

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
historia = model.fit(train_ds, validation_data=val_ds, epochs=20,
                     class_weight=class_weight, callbacks=callbacks('fase1'))

## 10. Fase 2
Ajustamos las últimas 20 capas con lr 1e-5, hasta 10 épocas; BatchNorm sigue congelada.

In [ ]:
base.trainable = True
for layer in base.layers:
    layer.trainable = False
for layer in base.layers[-20:]:
    layer.trainable = not isinstance(layer, BatchNormalization)
assert all(not layer.trainable for layer in base.layers if isinstance(layer, BatchNormalization))
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
historia2 = model.fit(train_ds, validation_data=val_ds, epochs=10,
                      class_weight=class_weight, callbacks=callbacks('fase2'))

## 11. Elegir la época
Elegimos la menor val_loss entre épocas con brecha ≤5 puntos, sin consultar prueba. Si ninguna cumple, conservamos el mínimo global y lo indicamos.

In [ ]:
tabla1 = pd.DataFrame(historia.history).assign(fase='fase1')
tabla2 = pd.DataFrame(historia2.history).assign(fase='fase2')
tabla1['epoca_fase'] = np.arange(1, len(tabla1) + 1)
tabla2['epoca_fase'] = np.arange(1, len(tabla2) + 1)
historial = pd.concat([tabla1, tabla2], ignore_index=True)
historial['epoca'] = np.arange(1, len(historial) + 1)
historial['brecha_accuracy'] = abs(historial.train_eval_accuracy - historial.val_accuracy)
candidatas = historial[historial.brecha_accuracy <= brecha_maxima]
cumple_brecha = not candidatas.empty
mejor = (candidatas if cumple_brecha else historial).sort_values('val_loss', kind='stable').iloc[0]
sufijo = 'brecha' if cumple_brecha else 'mejor'
model = tf.keras.models.load_model(carpeta_resultados / f"{mejor['fase']}_{sufijo}.keras")
print('Fase:', mejor['fase'], 'Época:', int(mejor.epoca),
      'Época de fase:', int(mejor.epoca_fase))
print('val_loss:', mejor.val_loss)
if not cumple_brecha:
    print('Ninguna época cumplió la brecha de 5 puntos.')
historial.to_csv(carpeta_resultados / 'historial.csv', index=False)

## 12. Curvas
La línea vertical marca la época elegida; train se evalúa sin aumento ni dropout.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(historial.epoca, historial.train_eval_accuracy, label='Train evaluado')
axes[0].plot(historial.epoca, historial.val_accuracy, label='Validación')
axes[0].set(title='Accuracy por época', xlabel='Época', ylabel='Accuracy')
axes[1].plot(historial.epoca, historial.train_eval_loss, label='Train evaluado')
axes[1].plot(historial.epoca, historial.val_loss, label='Validación')
axes[1].set(title='Pérdida por época', xlabel='Época', ylabel='Loss')
for ax in axes:
    ax.axvline(mejor.epoca, color='black', linestyle='--', label='Época elegida')
    ax.axvspan(len(tabla1) + 0.5, len(historial) + 0.5, alpha=0.08, color='gray')
    ax.legend()
plt.tight_layout()
plt.savefig(carpeta_resultados / 'curvas.png', dpi=150)
plt.show()

## 13. Prueba final
Evaluamos una vez elegido el modelo; estos datos no deciden la fase ni la época.

In [ ]:
loss, acc_keras = model.evaluate(test_ds, verbose=0)
probs = model.predict(test_ds, verbose=1)
predichas = np.argmax(probs, axis=1)
resultados = {
    'accuracy': accuracy_score(y_test, predichas),
    'precision_macro': precision_score(y_test, predichas, average='macro', zero_division=0),
    'recall_macro': recall_score(y_test, predichas, average='macro', zero_division=0),
    'f1_macro': f1_score(y_test, predichas, average='macro', zero_division=0),
    'loss_test': loss,
    'muestras_test': len(y_test),
}
print(pd.Series(resultados))
reporte = classification_report(y_test, predichas, labels=range(len(clases)),
                                target_names=clases, output_dict=True, zero_division=0)
print(classification_report(y_test, predichas, labels=range(len(clases)),
                            target_names=clases, zero_division=0))
pd.Series(resultados, name='valor').to_csv(carpeta_resultados / 'metricas.csv')
pd.DataFrame(reporte).transpose().to_csv(carpeta_resultados / 'metricas_por_clase.csv')
cm = confusion_matrix(y_test, predichas, labels=range(len(clases)))
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=clases).plot(ax=ax, cmap='Blues', xticks_rotation=25)
plt.title('Matriz de confusión (prueba)')
plt.tight_layout()
plt.savefig(carpeta_resultados / 'matriz_confusion.png', dpi=150)
plt.show()

## 14. Diagnóstico
Punto dulce exige brecha ≤5 puntos, prueba a ±3 de validación y accuracy ≥0.80. El 0.80 es una referencia ajustable; no usamos prueba para volver a elegir el modelo.

In [ ]:
acc_train = float(mejor.train_eval_accuracy)
acc_val = float(mejor.val_accuracy)
acc_test = float(resultados['accuracy'])
brecha_acc = abs(acc_train - acc_val)
brecha_loss = float(mejor.val_loss - mejor.train_eval_loss)
diferencia_test = abs(acc_test - acc_val)
if max(acc_train, acc_val) < accuracy_minima:
    veredicto = 'subajuste'
elif brecha_acc > brecha_maxima or diferencia_test > 0.03:
    veredicto = 'sobreajuste'
elif min(acc_train, acc_val, acc_test) < accuracy_minima:
    veredicto = 'subajuste'
else:
    veredicto = 'punto dulce'
print('Mejor época elegible:' if cumple_brecha else 'Mejor época global:', int(mejor.epoca))
print(f'Accuracy train: {acc_train:.4f} | validación: {acc_val:.4f} | prueba: {acc_test:.4f}')
print(f'Brecha accuracy absoluta: {100 * brecha_acc:.2f} puntos')
print(f'Brecha loss (validación - train): {brecha_loss:.4f}')
print(f'Diferencia prueba-validación: {100 * diferencia_test:.2f} puntos')
print('Veredicto:', veredicto)
print('Es un diagnóstico por umbrales; las curvas ayudan a revisar la causa.')
diagnostico = {
    'fase': str(mejor.fase), 'epoca': int(mejor.epoca),
    'epoca_fase': int(mejor.epoca_fase), 'val_loss': float(mejor.val_loss),
    'accuracy_train': acc_train, 'accuracy_val': acc_val, 'accuracy_test': acc_test,
    'brecha_accuracy': brecha_acc, 'brecha_loss': brecha_loss,
    'diferencia_test_val': diferencia_test, 'veredicto': veredicto,
}
(carpeta_resultados / 'diagnostico.json').write_text(
    json.dumps(diagnostico, ensure_ascii=False, indent=2), encoding='utf-8')

## 15. Qué ajustar
Subajuste: baja dropout a 0.3 o aumenta épocas si val_loss sigue bajando. Sobreajuste: sube L2 a 3e-4 o dropout a 0.5; punto dulce: conserva los parámetros.

## 16. Probar un dibujo
Vemos las probabilidades de una imagen de prueba.

In [ ]:
img_path = x_test[0]
img = image.load_img(img_path, target_size=target_size, interpolation='bilinear')
x = image.img_to_array(img)
x = preprocess_input(np.expand_dims(x, axis=0))
preds = model.predict(x, verbose=0)
print(pd.DataFrame({'clase': clases, 'probabilidad': preds[0]}).sort_values(
    'probabilidad', ascending=False))
print('Etiqueta real:', clases[y_test[0]])
plt.imshow(image.load_img(img_path))
plt.axis('off')
plt.show()

## 17. Guardar y descargar
Copia modelo_voto.keras y classes.json en models/ y reinicia Flask.

In [ ]:
model.save(carpeta_resultados / 'modelo_voto.keras')
(carpeta_resultados / 'classes.json').write_text(
    json.dumps(clases, ensure_ascii=False, indent=2), encoding='utf-8')
files.download(str(carpeta_resultados / 'modelo_voto.keras'))
files.download(str(carpeta_resultados / 'classes.json'))

## 18. Conclusiones